# Sampling and Monte Carlo Methods

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 03.08 |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/03_Probability_and_Statistics/09_sampling_and_monte_carlo.ipynb)

---

## 🎯 Learning Objective

Learn rejection sampling, importance sampling, and MCMC/Gibbs sampling as ways to draw samples from distributions that are hard to sample directly.

---

## 📐 Theory

Most interesting distributions can't be sampled directly by inverting their CDF (`03.02`) —
either the CDF has no closed form, or the distribution is only known up to an unknown
normalizing constant (a very common situation for Bayesian posteriors, `03.05`). This notebook
covers the general-purpose sampling algorithms that handle exactly this case.

### Rejection sampling

To sample from a target density $f(x)$ that's hard to sample directly, pick a **proposal**
distribution $q(x)$ that's easy to sample from and a constant $M$ such that $f(x) \le M\,q(x)$
for every $x$ (an *envelope*). Then: draw $x\sim q$, draw $u\sim\text{Uniform}(0,1)$, and
**accept** $x$ only if $u \le f(x)/(M q(x))$; otherwise reject and try again. Accepted samples
are exactly distributed as $f$ — accepting with probability proportional to how much of the
envelope $f$ fills at that $x$ corrects the proposal's shape into the target's shape. The
**acceptance rate** is $1/M$, so a loose envelope (large $M$) wastes most proposals — a central
weakness in high dimensions, where a tight envelope becomes almost impossible to construct.

### Importance sampling

Rather than reshaping a proposal into a target by discarding samples, **importance sampling**
keeps every sample and *reweights* it. To estimate $\mathbb{E}_f[h(X)] = \int h(x) f(x)\,dx$
using only samples from a proposal $q$:

$$\mathbb{E}_f[h(X)] = \int h(x) \frac{f(x)}{q(x)} q(x)\,dx = \mathbb{E}_q\left[h(X)\,w(X)\right], \qquad w(x) = \frac{f(x)}{q(x)},$$

where $w(x)$ is the **importance weight**. Draw $x_i\sim q$, and estimate the expectation as
$\frac{1}{n}\sum_i h(x_i)w(x_i)$ — no samples wasted, but the estimator's variance blows up
whenever $q$ is small where $f$ is large (a few huge weights dominate the sum), so choosing $q$
close to $f$ still matters enormously.

### Markov Chain Monte Carlo (MCMC)

When even evaluating $f$ up to its normalizing constant is all you have (the norm situation for
posteriors $p(\theta\mid D)\propto P(D\mid\theta)P(\theta)$, where the denominator $P(D)$ is an
intractable integral), **MCMC** builds a Markov chain whose *stationary distribution* is the
target $f$, then simulates it and collects visited states as (correlated) samples.
**Metropolis-Hastings** is the general recipe: from current state $x_t$, propose $x'\sim
q(\cdot\mid x_t)$, and accept it as $x_{t+1}$ with probability

$$\alpha = \min\left(1, \frac{f(x')\,q(x_t\mid x')}{f(x_t)\,q(x'\mid x_t)}\right);$$

otherwise stay at $x_{t+1}=x_t$. Because $f$ only ever appears as a *ratio* $f(x')/f(x_t)$, any
unknown normalizing constant cancels exactly — the algorithm never needs to know it. This
accept/reject step is what guarantees the chain's long-run visit frequencies match $f$.

### Gibbs sampling

**Gibbs sampling** is a special case of Metropolis-Hastings for multivariate targets, with an
acceptance probability that's always exactly 1. To sample $(X,Y)\sim f(x,y)$, alternate:
$x_{t+1}\sim f(x\mid y_t)$, then $y_{t+1}\sim f(y\mid x_{t+1})$ — cycling through each variable's
full conditional (`03.04`) with every other variable held fixed. It needs those conditionals to
be directly sampleable, but when they are (common for conjugate Bayesian models, `03.05`-`03.06`),
it needs no proposal tuning at all.

### Why any of this converges

All of these are **Monte Carlo methods**: they replace an intractable integral/expectation with
a sample average, justified by the **Law of Large Numbers** — the average of $n$ iid (or, for
MCMC, ergodic-chain) samples converges to the truth as $n\to\infty$. MCMC's added subtlety is
that consecutive samples are correlated, so it needs a "burn-in" period and yields a smaller
*effective* sample size than its raw count.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Rejection sampling made visible: proposal samples scattered under an envelope, colored by
accept/reject, so the accepted points visibly trace out the target density's shape.

In [ ]:
# Rejection sampling from a bimodal target f(x) using a scaled Uniform(-4,4) envelope: every
# proposed point is a dot, colored by whether the acceptance test kept it.
rng = np.random.default_rng(0)

def target_density(x):
    # An unnormalized bimodal density -- deliberately NOT a named scipy distribution, to make
    # the point that rejection sampling only needs to EVALUATE f, not know its normalizer or CDF
    return 0.6 * np.exp(-0.5 * ((x - 2) / 0.7) ** 2) + 0.4 * np.exp(-0.5 * ((x + 2) / 1.0) ** 2)

x_range = np.linspace(-6, 6, 500)
M = 1.05 * target_density(x_range).max()  # envelope height: proposal density * M must dominate f everywhere
proposal_low, proposal_high = -6, 6
envelope_height = M / (proposal_high - proposal_low)  # since Uniform's density is 1/(high-low)

n_proposals = 3000
proposals_x = rng.uniform(proposal_low, proposal_high, n_proposals)
proposals_u = rng.uniform(0, 1, n_proposals) * envelope_height  # height under the flat envelope
accept_mask = proposals_u <= target_density(proposals_x) / (proposal_high - proposal_low)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(proposals_x[~accept_mask], proposals_u[~accept_mask], s=4, alpha=0.3, color="gray", label="rejected")
axes[0].scatter(proposals_x[accept_mask], proposals_u[accept_mask], s=4, alpha=0.6, color="#C44E52", label="accepted")
axes[0].plot(x_range, target_density(x_range) / (proposal_high - proposal_low), color="#4C72B0", lw=2, label="f(x) [rescaled]")
axes[0].axhline(envelope_height, color="black", ls="--", lw=1, label="envelope M*q(x)")
axes[0].set_title("Rejection sampling: accept iff the proposal\nfalls UNDER the target curve"); axes[0].legend(fontsize=8)

axes[1].hist(proposals_x[accept_mask], bins=50, density=True, color="#C44E52", alpha=0.6, label="accepted samples")


def trapz_1d(values, x):
    # Hand-rolled trapezoidal rule -- avoids relying on np.trapz/np.trapezoid across numpy versions.
    widths = np.diff(x)
    return np.sum(widths * (values[:-1] + values[1:]) / 2)


normalizer = trapz_1d(target_density(x_range), x_range)
axes[1].plot(x_range, target_density(x_range) / normalizer, "k--", lw=2, label="true (normalized) f(x)")
axes[1].set_title("Accepted samples reproduce the target density"); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"Acceptance rate: {accept_mask.mean():.3f}  (theoretical ~ 1/M-ish, set by how tight the envelope is)")
print(f"Number of accepted samples: {accept_mask.sum()} out of {n_proposals} proposals")

## 🐍 Implementation from Scratch

We implement importance sampling for an expectation, Metropolis-Hastings MCMC for a 2D target
with an unknown normalizing constant, and Gibbs sampling for a bivariate Gaussian using its
known conditional distributions — comparing every result against ground truth.

In [ ]:
# --- Importance sampling: estimate E_f[X] for f=Normal(3,1), using ONLY samples from a
# different proposal q=Normal(0,2), reweighted by w(x) = f(x)/q(x) ---
rng = np.random.default_rng(0)
f_dist, q_dist = stats.norm(3, 1), stats.norm(0, 2)   # TARGET vs. PROPOSAL

n_samples = 20_000
samples_q = q_dist.rvs(size=n_samples, random_state=0)
weights = f_dist.pdf(samples_q) / q_dist.pdf(samples_q)
importance_estimate = np.sum(samples_q * weights) / np.sum(weights)  # self-normalized IS

print(f"True E_f[X] = {f_dist.mean():.4f}")
print(f"Importance sampling estimate (samples drawn from a DIFFERENT distribution): {importance_estimate:.4f}")
print(f"Effective sample size: {np.sum(weights)**2 / np.sum(weights**2):.1f} out of {n_samples}")

# --- Metropolis-Hastings MCMC: sample a target known only up to an unknown constant ---
def unnormalized_target(xy):
    # Correlated 2D Gaussian shape, scaled by an ARBITRARY constant (37.2): MCMC only ever
    # uses the RATIO target(x')/target(x), so an unknown normalizer cancels automatically.
    x, y, rho = xy[0], xy[1], 0.8
    return 37.2 * np.exp(-(x**2 - 2 * rho * x * y + y**2) / (2 * (1 - rho**2)))

def metropolis_hastings(target_unnorm, init, n_steps, proposal_std, rng):
    current = np.array(init, dtype=float)
    samples, n_accepted = np.empty((n_steps, 2)), 0
    for t in range(n_steps):
        proposal = current + rng.normal(scale=proposal_std, size=2)  # symmetric -> q cancels in alpha
        if rng.uniform() < min(1.0, target_unnorm(proposal) / target_unnorm(current)):
            current = proposal
            n_accepted += 1
        samples[t] = current
    return samples, n_accepted / n_steps

mh_samples, accept_rate = metropolis_hastings(unnormalized_target, init=[0, 0], n_steps=20_000,
                                               proposal_std=0.6, rng=rng)
burn_in = 2000
mh_kept = mh_samples[burn_in:]
print(f"\nMCMC acceptance rate: {accept_rate:.3f}")
print(f"MCMC sample mean (should be ~[0,0]): {mh_kept.mean(axis=0).round(3)}")
print(f"MCMC sample correlation (should be ~0.8): {np.corrcoef(mh_kept.T)[0,1]:.3f}")

# --- Gibbs sampling: full conditionals of a bivariate Gaussian are KNOWN Gaussians, so every
# step samples EXACTLY (acceptance probability always 1, no rejection needed) ---
def gibbs_bivariate_gaussian(rho, n_steps, rng):
    x, y, cond_std = 0.0, 0.0, np.sqrt(1 - rho**2)
    samples = np.empty((n_steps, 2))
    for t in range(n_steps):
        x = rng.normal(rho * y, cond_std)   # x | y ~ N(rho*y, 1-rho^2)
        y = rng.normal(rho * x, cond_std)   # y | x ~ N(rho*x, 1-rho^2)
        samples[t] = (x, y)
    return samples

gibbs_kept = gibbs_bivariate_gaussian(rho=0.8, n_steps=20_000, rng=rng)[burn_in:]
print(f"\nGibbs sample mean (should be ~[0,0]):      {gibbs_kept.mean(axis=0).round(3)}")
print(f"Gibbs sample correlation (should be ~0.8): {np.corrcoef(gibbs_kept.T)[0,1]:.3f}")

## 🤖 Why This Matters for AI

Every one of these techniques shows up directly in modern ML: **variational inference** turns
an intractable Bayesian posterior into an optimization problem, but is validated against MCMC
ground truth; **Bayes by Backprop** (`03.11`) treats network weights as a distribution sampled
during training; and every time a language model generates text, it's **sampling from a
categorical distribution over tokens** — where temperature, top-k, and nucleus (top-p) sampling
are all just different ways of reshaping that distribution before drawing from it, demonstrated
concretely below.

---

In [ ]:
# LLM decoding strategies: temperature scaling, top-k, and nucleus (top-p) sampling all
# RESHAPE a categorical distribution (03.02, 03.06) over the vocabulary before drawing from it.
rng = np.random.default_rng(0)
vocab = [f"tok_{i}" for i in range(10)]
logits = np.array([4.0, 3.5, 2.0, 1.8, 1.0, 0.5, 0.2, -0.5, -1.0, -2.0])  # one skewed logit vector

def softmax(z):
    z = z - z.max()
    return np.exp(z) / np.exp(z).sum()

def temperature_scale(logits, T):
    # Dividing logits by T before softmax: T<1 sharpens the distribution (more confident,
    # closer to argmax), T>1 flattens it (more random) -- same logits, reshaped distribution.
    return softmax(logits / T)

def top_k_filter(probs, k):
    # Zero out everything except the k highest-probability tokens, then renormalize the rest.
    keep_idx = np.argsort(probs)[-k:]
    filtered = np.zeros_like(probs)
    filtered[keep_idx] = probs[keep_idx]
    return filtered / filtered.sum()

def nucleus_filter(probs, p):
    # Keep the SMALLEST set of highest-probability tokens whose cumulative mass exceeds p,
    # then renormalize -- the set size adapts to how "peaked" or "flat" the distribution is.
    order = np.argsort(probs)[::-1]
    cumulative = np.cumsum(probs[order])
    cutoff = np.searchsorted(cumulative, p) + 1
    keep_idx = order[:cutoff]
    filtered = np.zeros_like(probs)
    filtered[keep_idx] = probs[keep_idx]
    return filtered / filtered.sum(), cutoff

base_probs = softmax(logits)
print("Base softmax probabilities (T=1, no filtering):")
print(np.round(base_probs, 4))

print("\n--- Temperature scaling reshapes concentration, without changing WHICH token ranks highest ---")
for T in [0.3, 1.0, 2.0]:
    probs_T = temperature_scale(logits, T)
    entropy_T = -np.sum(probs_T * np.log(probs_T + 1e-12))  # a concentration summary, formalized in 05.01
    print(f"  T={T:.1f}: top prob={probs_T.max():.4f}, entropy={entropy_T:.4f}, "
          f"argmax token={vocab[np.argmax(probs_T)]}")
print("  (lower T -> lower entropy, more concentrated on the top token; higher T -> flatter, more random)")

print("\n--- top-k keeps a FIXED candidate count regardless of how peaked the distribution is ---")
for k in [1, 3, 10]:
    probs_k = top_k_filter(base_probs, k)
    n_nonzero = np.sum(probs_k > 0)
    print(f"  k={k}: kept {n_nonzero} tokens, renormalized probs sum to {probs_k.sum():.4f}")

print("\n--- nucleus (top-p) keeps a VARIABLE candidate count, adapting to the distribution shape ---")
for p in [0.5, 0.9, 0.99]:
    probs_p, n_kept = nucleus_filter(base_probs, p)
    print(f"  p={p}: kept {n_kept} tokens (adaptive, not fixed) to cover >= {p:.0%} of the mass")

print("\nSampling 5 tokens from the T=0.5, top-k=3 reshaped distribution:")
final_probs = top_k_filter(temperature_scale(logits, 0.5), k=3)
sampled_tokens = rng.choice(vocab, size=5, p=final_probs)
print(" ", list(sampled_tokens))

## 🏋️ Exercises

### Warm-up
1. In the rejection sampling cell, reduce the envelope constant `M` (e.g. `1.01` instead of
   `1.05`) and rerun. Confirm the acceptance rate changes as predicted, and explain why `M`
   must never be smaller than `1.0` relative to the true maximum ratio.

### Practice
2. Modify `metropolis_hastings` to track and return the FULL chain (not just after burn-in),
   then plot the sample path of $x_t$ over the first 200 steps. Identify roughly where the
   chain stops reflecting its start point and settles into sampling the target — the informal
   way "burn-in" length is often chosen in practice.

### Challenge
3. Combine temperature scaling AND nucleus sampling: apply `temperature_scale(logits, T=1.5)`,
   then `nucleus_filter(..., p=0.9)` to the result. Compare the tokens kept and resulting
   entropy to nucleus sampling alone (T=1). Explain why a language model API exposing both
   `temperature` and `top_p` lets users combine them, rather than treating them as redundant.

---

## 📚 Further Reading
- Robert & Casella, *Monte Carlo Statistical Methods*
- Holtzman et al., ["The Curious Case of Neural Text Degeneration"](https://arxiv.org/abs/1904.09751) (nucleus sampling)
- MacKay, *Information Theory, Inference, and Learning Algorithms*, Ch. 29–30 (Monte Carlo Methods)

---

*Next notebook: [Probabilistic Models in Machine Learning](10_probabilistic_models_in_ml.ipynb)*